# 02 — Content-Based Filtering (TF-IDF + Cosine Similarity)

**H&M Personalized Fashion Recommendations**

BTEC L6 Unit 2 — Capstone Project · Lola Toirxonova (ID 220062)

---

## Algorithm

1. Build a text corpus per article from `prod_name`, `detail_desc`, `product_type_name`, `colour_group_name`, `department_name`, `index_group_name`.
2. Vectorise with **TF-IDF** (`scikit-learn`).
3. For each user, compute a **user profile vector** = average of TF-IDF vectors of items they purchased in the train set.
4. Score every item by cosine similarity to the user profile; remove items already purchased; return top-N.

## Why this baseline matters

Content-based filtering is the **only one of the four required algorithms** that can recommend cold items (new products with zero purchase history) because it relies on item metadata, not interactions. This makes it valuable both as a standalone baseline and as the content side of the **hybrid model** (Notebook 04).

## Criteria addressed

P4 (implementation), P5 + M3 (data analysis methods), M4 (compare and visualise), D3 (cold-start segmented metrics).

## Setup

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix

from src import data as dataio
from src import metrics as metricslib

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

OUTPUT_DIR = REPO_ROOT / 'outputs' / 'content_based'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = REPO_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
TOP_K = 10
LAST_MONTHS = 6  # last 6 months — matches H&M competition protocol
np.random.seed(RANDOM_SEED)

## 1. Load data

In [2]:
articles = dataio.load_articles()
transactions = dataio.load_transactions(last_months=LAST_MONTHS)
print(f'Articles: {len(articles):,}   |   Transactions sample: {len(transactions):,}')

Articles: 105,542   |   Transactions sample: 8,150,956


In [3]:
train, test = dataio.time_based_split(transactions, cutoff_days=7)
print(f'Train: {len(train):,}   |   Test: {len(test):,}')
print(f'Cutoff: {train["t_dat"].max().date()}  ->  test from {test["t_dat"].min().date()}')

Train: 7,910,645   |   Test: 240,311
Cutoff: 2020-09-15  ->  test from 2020-09-16


## 2. Build the text corpus

Concatenate the meaningful text columns. Missing values become empty strings.

In [4]:
text_cols = ['prod_name', 'detail_desc', 'product_type_name',
             'product_group_name', 'colour_group_name',
             'department_name', 'index_group_name']

articles_text = articles[['article_id'] + [c for c in text_cols if c in articles.columns]].copy()
articles_text = articles_text.fillna('')
articles_text['corpus'] = articles_text[[c for c in text_cols if c in articles_text.columns]].astype(str).agg(' '.join, axis=1)

print(f'Corpus rows: {len(articles_text):,}')
print('\nExample corpus entries (first 3):')
for i in range(3):
    print(f'\n[{i}] {articles_text["corpus"].iloc[i][:200]}...')

Corpus rows: 105,542

Example corpus entries (first 3):

[0] Strap top Jersey top with narrow shoulder straps. Vest top Garment Upper body Black Jersey Basic Ladieswear...

[1] Strap top Jersey top with narrow shoulder straps. Vest top Garment Upper body White Jersey Basic Ladieswear...

[2] Strap top (1) Jersey top with narrow shoulder straps. Vest top Garment Upper body Off White Jersey Basic Ladieswear...


## 3. TF-IDF vectorisation

In [5]:
vectorizer = TfidfVectorizer(
    max_features=5_000,
    ngram_range=(1, 2),
    min_df=2,
    stop_words='english',
    lowercase=True,
    sublinear_tf=True,
)
item_tfidf = vectorizer.fit_transform(articles_text['corpus'])
item_tfidf = normalize(item_tfidf, norm='l2', axis=1)
print(f'Item-TFIDF matrix shape: {item_tfidf.shape}   (items × features)')
print(f'Sparsity: {1 - item_tfidf.nnz / (item_tfidf.shape[0] * item_tfidf.shape[1]):.4%}')

Item-TFIDF matrix shape: (105542, 5000)   (items × features)
Sparsity: 99.1863%


In [6]:
# Article-id -> row index lookup (used to map purchases -> tf-idf rows)
item_id_to_row = {aid: i for i, aid in enumerate(articles_text['article_id'].values)}
row_to_item_id = articles_text['article_id'].values

## 4. Build user profile vectors

Each user's profile = mean of TF-IDF vectors of items purchased in the train set.

In [7]:
def build_user_profiles(train_df: pd.DataFrame, item_tfidf: csr_matrix, item_id_to_row: dict) -> tuple:
    """Return (user_profiles_csr, user_ids_ordered)."""
    # Map article ids to row indices, drop interactions referring to unknown items
    train_df = train_df.copy()
    train_df['item_row'] = train_df['article_id'].map(item_id_to_row)
    train_df = train_df.dropna(subset=['item_row'])
    train_df['item_row'] = train_df['item_row'].astype(int)

    users = pd.Index(train_df['customer_id'].unique(), name='customer_id')
    user_pos = users.get_indexer(train_df['customer_id'])
    n_users = len(users)
    n_features = item_tfidf.shape[1]

    # Count interactions per user for averaging
    interaction_count = np.bincount(user_pos, minlength=n_users).astype(np.float32)
    interaction_count[interaction_count == 0] = 1  # avoid div-by-zero

    # Sum item vectors per user
    row_select = csr_matrix(
        (np.ones(len(train_df), dtype=np.float32), (user_pos, train_df['item_row'].values)),
        shape=(n_users, item_tfidf.shape[0]),
    )
    user_profiles = row_select @ item_tfidf  # (users × items) @ (items × features) -> (users × features)
    # Convert to dense scaling — interaction_count is dense (n_users,)
    from scipy.sparse import diags
    inv_counts = diags(1.0 / interaction_count)
    user_profiles = inv_counts @ user_profiles
    user_profiles = normalize(user_profiles, norm='l2', axis=1)
    return user_profiles, users

user_profiles, user_index = build_user_profiles(train, item_tfidf, item_id_to_row)
print(f'User profiles shape: {user_profiles.shape}   ({user_profiles.shape[0]:,} users)')

User profiles shape: (734555, 5000)   (734,555 users)


## 5. Recommend top-N items

For each evaluated user: score items = user_profile @ item_tfidf.T, mask out items already in train, take top-K.

In [8]:
def user_seen_items(train_df: pd.DataFrame) -> dict:
    """customer_id -> set of article_ids already in train."""
    return train_df.groupby('customer_id')['article_id'].apply(set).to_dict()

seen = user_seen_items(train)
print(f'Users with at least one train interaction: {len(seen):,}')

Users with at least one train interaction: 734,555


In [9]:
def recommend_for_users(
    user_profiles, user_index, item_tfidf, row_to_item_id, seen_items,
    user_ids: list, k: int = 10,
) -> dict:
    """Return {customer_id: [article_id, ...]} of length k for each user that has a profile."""
    recommendations = {}
    user_id_to_row = {u: i for i, u in enumerate(user_index)}
    for u in user_ids:
        if u not in user_id_to_row:
            continue  # cold user — handled separately
        profile = user_profiles[user_id_to_row[u]]
        if profile.nnz == 0:
            continue
        scores = (profile @ item_tfidf.T).toarray().ravel()
        # Mask items already seen
        already_seen = seen_items.get(u, set())
        if already_seen:
            seen_rows = [item_id_to_row[a] for a in already_seen if a in item_id_to_row]
            scores[seen_rows] = -np.inf
        top_idx = np.argpartition(-scores, k)[:k]
        top_idx = top_idx[np.argsort(-scores[top_idx])]
        recommendations[u] = [row_to_item_id[i] for i in top_idx]
    return recommendations

## 6. Evaluate on the test set

In [10]:
# Ground truth: customer_id -> set of article_ids purchased in test window
ground_truth = test.groupby('customer_id')['article_id'].apply(set).to_dict()
print(f'Users with test purchases: {len(ground_truth):,}')

# Evaluate only users who appear in both train (have a profile) and test (have ground truth)
warm_test_users = [u for u in ground_truth if u in seen]
cold_test_users = [u for u in ground_truth if u not in seen]
print(f'Warm test users: {len(warm_test_users):,}   |   Cold test users: {len(cold_test_users):,}')

Users with test purchases: 68,984
Warm test users: 56,857   |   Cold test users: 12,127


In [11]:
# Cap evaluation set if very large, for runtime
EVAL_USER_CAP = 5_000
if len(warm_test_users) > EVAL_USER_CAP:
    rng = np.random.default_rng(RANDOM_SEED)
    warm_eval = list(rng.choice(warm_test_users, size=EVAL_USER_CAP, replace=False))
else:
    warm_eval = warm_test_users
print(f'Evaluating on {len(warm_eval):,} warm users')

recommendations_warm = recommend_for_users(
    user_profiles, user_index, item_tfidf, row_to_item_id, seen,
    warm_eval, k=TOP_K,
)
print(f'Generated recommendations for {len(recommendations_warm):,} users')

Evaluating on 5,000 warm users


Generated recommendations for 5,000 users


In [12]:
warm_metrics = metricslib.evaluate(recommendations_warm, ground_truth, k=TOP_K)
warm_metrics

{'Precision@10': 0.00204,
 'Recall@10': 0.00764516095016095,
 'HitRate@10': 0.0178,
 'MAP@10': 0.0037792328042328045,
 'NDCG@10': 0.0057171453451400605,
 'users_evaluated': 5000}

## 7. Cold-start segmented evaluation (D3)

Content-based filtering cannot make personalised recommendations for cold users (no profile). Report this honestly — it is a strength of D3 to acknowledge the algorithm's limits.

In [13]:
print(f'Cold users with test purchases: {len(cold_test_users):,}')
print('Content-based filtering returns no recommendation for cold users by construction.')
print('In the hybrid model (Notebook 04) this is handled by a popularity fallback.')

Cold users with test purchases: 12,127
Content-based filtering returns no recommendation for cold users by construction.
In the hybrid model (Notebook 04) this is handled by a popularity fallback.


## 8. Save model artefacts

In [14]:
with open(MODEL_DIR / 'content_based_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

from scipy.sparse import save_npz
save_npz(MODEL_DIR / 'content_based_item_tfidf.npz', item_tfidf)

results = {
    'model': 'content_based_tfidf',
    'sample_size_transactions': len(transactions),
    'top_k': TOP_K,
    'eval_warm_users': len(warm_eval),
    'eval_cold_users': len(cold_test_users),
    'metrics_warm': warm_metrics,
    'metrics_cold': 'not applicable — algorithm cannot personalise cold users',
}
with open(OUTPUT_DIR / 'results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
results

{'model': 'content_based_tfidf',
 'sample_size_transactions': 8150956,
 'top_k': 10,
 'eval_warm_users': 5000,
 'eval_cold_users': 12127,
 'metrics_warm': {'Precision@10': 0.00204,
  'Recall@10': 0.00764516095016095,
  'HitRate@10': 0.0178,
  'MAP@10': 0.0037792328042328045,
  'NDCG@10': 0.0057171453451400605,
  'users_evaluated': 5000},
 'metrics_cold': 'not applicable — algorithm cannot personalise cold users'}

## 9. Qualitative spot-check

Pick 3 random users and show what they bought + what the model recommends. This is the kind of artefact that goes into the report's Findings chapter.

In [15]:
def describe_items(item_ids, articles_df, max_show=5):
    cols = [c for c in ['article_id', 'prod_name', 'product_type_name', 'colour_group_name'] if c in articles_df.columns]
    return articles_df.set_index('article_id').loc[
        [i for i in item_ids if i in articles_df['article_id'].values][:max_show],
        [c for c in cols if c != 'article_id']
    ]

rng = np.random.default_rng(RANDOM_SEED)
spot_check_users = rng.choice(list(recommendations_warm.keys()), size=3, replace=False)
for u in spot_check_users:
    print(f'\n=== User {u} ===')
    print('\nPurchased in train (sample):')
    print(describe_items(list(seen.get(u, set()))[:5], articles))
    print('\nRecommended:')
    print(describe_items(recommendations_warm[u], articles))
    print('\nActual test purchases:')
    print(describe_items(list(ground_truth.get(u, set())), articles))


=== User 8d438716c814e573bb41442ca0f26fe6e99167b77b90a791c34bd9a4e1e53160 ===

Purchased in train (sample):
           prod_name product_type_name colour_group_name
article_id                                              
0637055003      Tuss             Dress        Light Blue

Recommended:
           prod_name product_type_name colour_group_name
article_id                                              
0637055001      Tuss             Dress         Dark Blue
0637055002      Tuss             Dress      Light Orange
0637055004      Tuss             Dress             Green
0606894001   Tuss OL             Dress               Red
0619623001  Cia maxi             Dress             Black

Actual test purchases:
                                prod_name product_type_name colour_group_name
article_id                                                                   
0918292001      STRONG HW seamless tights   Leggings/Tights             Black
0719655014        Greta Ch hipster ctn 3p  Underw

## Next steps

1. Note the **Precision@10 / Recall@10 / NDCG@10** numbers — they are the **content-based baseline** that every later model must beat.
2. Write 2–3 paragraphs in `docs/findings_content_based.md` with one figure (the spot-check) and the metrics table.
3. Update `logbook.md` for the week.
4. Move on to `03_collaborative_filtering.ipynb` — matrix factorisation with `implicit` (ALS) on the same train/test split for fair comparison.